In [ ]:
from pathlib import Path
import pandas as pd


project_root = Path.cwd().parent
file_path = project_root / "data" / "raw" / "retail_store_inventory.csv"

df = pd.read_csv(file_path)
df.head()

In [ ]:
# Le ponemos formato de fecha a nuestro Date, ya que por defecto viene como object

df["Date"] = pd.to_datetime(df["Date"])


In [10]:
#vamos a limpiar los registros de Demand Forecast que son negativos y dejar un copia para documentar la desicion

negative_demand = df[df["Demand Forecast"] < 0].copy()
print("Registros con demanda negativa: ", len(negative_demand))

negative_demand[[
    "Date",
    "Store ID",
    "Product ID",
    "Units Sold",
    "Demand Forecast"
]].head(10)


Registros con demanda negativa:  673


,Date,Store ID,Product ID,Units Sold,Demand Forecast
63,2022-01-01,S004,P0004,0,-2.40
141,2022-01-02,S003,P0002,2,-3.40
278,2022-01-03,S004,P0019,1,-3.91
511,2022-01-06,S001,P0012,1,-8.37
730,2022-01-08,S002,P0011,7,-2.99
844,2022-01-09,S003,P0005,4,-1.33
1011,2022-01-11,S001,P0012,3,-6.05
1107,2022-01-12,S001,P0008,0,-3.55
1149,2022-01-12,S003,P0010,6,-0.11
1180,2022-01-12,S005,P0001,6,-2.04


In [11]:
negative_demand[[
    "Units Sold",
    "Demand Forecast"
]].describe()


,Units Sold,Demand Forecast
count,673.000000,673.000000
mean,2.912333,-3.693730
std,2.533651,2.535978
min,0.000000,-9.990000
25%,1.000000,-5.470000
50%,2.000000,-3.380000
75%,5.000000,-1.540000
max,9.000000,-0.010000


In [ ]:
negative_demand["Date"].dt.to_period("M").value_counts().sort_index()

In [14]:
print("valores negativos:", (df["Demand Forecast"] < 0).sum())
print("valor mínimo antes:", df["Demand Forecast"].min())

valores negativos: 673
valor mínimo antes: -9.99


In [15]:
df["Demand Forecast"] = df["Demand Forecast"].clip(lower=0)

print("valor mínimo después:", df["Demand Forecast"].min())

valor mínimo después: 0.0


In [18]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month Name"] = df["Date"].dt.month_name()
df["Day"] = df["Date"].dt.day

In [26]:
df["Estimated Sales"] = (
    df["Units Sold"]
    * df["Price"]
    * (1 - df["Discount"] / 100)
)

In [28]:
print("filas:", len(df))
print("columnas:", len(df.columns))
print(df.columns.tolist())

filas: 73100
columnas: 20
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Month', 'Month Name', 'Day', 'Estimated Sales']


In [29]:
out_path = project_root / "data" / "processed" / "retail_store_inventory_clean.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)


In [31]:
df_processed = pd.read_csv(
    out_path,
    parse_dates=["Date"]
)

print("filas:", len(df_processed))
print("columnas:", len(df_processed.columns))
print("fecha mínima:", df_processed["Date"].min())
print("fecha máxima:", df_processed["Date"].max())
print("forecast negativo:", (df_processed["Demand Forecast"] < 0).sum())
print("valores nulos:", df_processed.isna().sum().sum())
print("duplicados:", df_processed.duplicated().sum())

filas: 73100
columnas: 20
fecha mínima: 2022-01-01 00:00:00
fecha máxima: 2024-01-01 00:00:00
forecast negativo: 0
valores nulos: 0
duplicados: 0
